In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pandas as pd

spark = (
    SparkSession.builder
    .enableHiveSupport()
    .appName("cu2")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .getOrCreate()
)
sql = spark.sql

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/22 16:19:24 WARN Utils: Service 'sparkDriver' could not bind on port 23527. Attempting port 23528.
26/06/22 16:19:24 WARN Utils: Service 'sparkDriver' could not bind on port 23528. Attempting port 23529.
26/06/22 16:19:34 WARN Utils: Service 'org.apache.spark.network.netty.NettyBlockTransferService' could not bind on port 23527. Attempting port 23528.
26/06/22 16:19:34 WARN Utils: Service 'org.apache.spark.network.netty.NettyBlockTransferService' could not bind on port 23528. Attempting port 23529.
26/06/22 16:19:34 WARN Utils: Service 'org.apache.spark.network.netty.NettyBlockTransferService' could not bind on port 23529. Attempting port 23530.


In [4]:
codes = pd.read_csv(
    "../data/Referentiel_CIM-10-20250108.csv", sep=";", header=1, dtype=str
)

codes = codes[["code", "libellé long", "code MCO/HAD"]].rename(
    columns={"libellé long": "definition", "code MCO/HAD": "mco_had"}
)
codes["code"] = codes["code"].astype(str).str.strip()
codes["mco_had"] = pd.to_numeric(codes["mco_had"], errors="coerce")

dp = codes[codes["mco_had"].isin([0])].drop("mco_had", axis=1)
das = codes[codes["mco_had"].isin([0,1,2])].drop("mco_had", axis=1)

dp=dp["code"].to_list()
das=das["code"].to_list()
mdp = [item for item in dp if item.startswith('Z')]

In [8]:
pd.to_pickle(dp, "../data/valid_labels_all_dp.pkl")
pd.to_pickle(das, "../data/valid_labels_all_das.pkl")
pd.to_pickle(mdp, "../data/valid_labels_all_mdp.pkl")